In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.cloud import bigquery
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from vertexai.language_models import TextEmbeddingModel
import vertexai

In [ ]:
def ensure_dataset(client: bigquery.Client, dataset_id: str, location: str = "us-central1"):
    ds_ref = bigquery.Dataset(f"{client.project}.{dataset_id}")
    try:
        client.get_dataset(ds_ref)
    except Exception:
        ds_ref.location = location
        client.create_dataset(ds_ref)
        print(f"Created dataset: {dataset_id} in {location}")

def nearest_neighbors_sql(table_id: str, topk: int = 10) -> str:
    return f"""
    DECLARE query_vec ARRAY<FLOAT64>;
    SET query_vec = @query_vec;

    SELECT
      patient_id,
      label,
      modality_text,
      VECTOR_DISTANCE(embedding, query_vec, 'COSINE') AS cosine_dist
    FROM `{table_id}`
    ORDER BY cosine_dist ASC
    LIMIT {topk};
    """


In [ ]:
clin_df = pd.read_csv('###')
clinc_text_df = pd.read_csv('###')
clinc_onehot = pd.read_csv('###')
label_df = pd.read_csv('###')
gen_df = pd.read_csv('###')

In [ ]:
PROJECT_ID = "###"
DATASET_ID = "###"

bq = bigquery.Client(project=PROJECT_ID)
job = bq.load_table_from_dataframe(clin_df, f"{PROJECT_ID}.{DATASET_ID}.DF_CLINIC")
job.result()
job = bq.load_table_from_dataframe(gen_df, f"{PROJECT_ID}.{DATASET_ID}.DF_GEN")
job.result()
job = bq.load_table_from_dataframe(label_df, f"{PROJECT_ID}.{DATASET_ID}.DF_LABEL")
job.result()
job = bq.load_table_from_dataframe(clinc_text_df, f"{PROJECT_ID}.{DATASET_ID}.DF_CLI_TEXT")
job.result()

In [ ]:
table_name = "DF_CLINIC"
ensure_dataset(bq,DATASET_ID)

table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
nearest_neighbors_sql(table_id, topk=5)